# Notebook 02 — Transformação e Métricas OEE Energético
**Projeto:** Steel Plant Analytics
**Autor:** Ariel Marquezin
**Stack:** DuckDB · pandas · numpy · PyArrow

In [ ]:
!pip install duckdb pandas numpy pyarrow azure-storage-blob -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
from azure.storage.blob import BlobServiceClient

BRONZE_PATH = '/content/drive/MyDrive/steel/bronze/'
GOLD_PATH   = '/content/drive/MyDrive/steel/gold/'
os.makedirs(GOLD_PATH, exist_ok=True)
AZURE_CONN  = os.environ.get('AZURE_STORAGE_CONNECTION_STRING', '')
CONTAINER   = 'steel-gold'
print('Ambiente OK')

In [ ]:
# Leitura Bronze via DuckDB
con = duckdb.connect()
bronze_file = os.path.join(BRONZE_PATH, 'steel_bronze.parquet')
con.execute(f"CREATE VIEW steel AS SELECT * FROM parquet_scan('{bronze_file}')")
df = con.execute('SELECT * FROM steel').df()
print(f'Bronze carregado: {len(df):,} registros')
print(df.columns.tolist())

In [ ]:
# Identifica colunas dinamicamente
energy_col   = [c for c in df.columns if 'kwh' in c.lower() or 'consumption' in c.lower() or 'usage' in c.lower()][0]
reactive_cols = [c for c in df.columns if 'reactive' in c.lower() or 'lagging' in c.lower() or 'leading' in c.lower()]
load_col      = [c for c in df.columns if 'load' in c.lower() or 'type' in c.lower()]
co2_col       = [c for c in df.columns if 'co2' in c.lower()]
print(f'Coluna energia   : {energy_col}')
print(f'Colunas reativas : {reactive_cols}')
print(f'Coluna carga     : {load_col}')
print(f'Coluna CO2       : {co2_col}')

In [ ]:
# Metricas por Turno
df_turno = df.groupby('turno').agg(
    total_registros   = (energy_col, 'count'),
    consumo_total_kwh = (energy_col, 'sum'),
    consumo_medio_kwh = (energy_col, 'mean'),
    consumo_max_kwh   = (energy_col, 'max'),
    consumo_min_kwh   = (energy_col, 'min'),
    desvio_kwh        = (energy_col, 'std'),
).reset_index()
df_turno['pct_consumo_total'] = (df_turno['consumo_total_kwh'] / df_turno['consumo_total_kwh'].sum() * 100).round(2)
df_turno['consumo_por_hora']  = (df_turno['consumo_total_kwh'] / (df_turno['total_registros'] * 0.25)).round(2)
print(df_turno[['turno','consumo_total_kwh','consumo_medio_kwh','pct_consumo_total']].to_string())

In [ ]:
# OEE Energetico por Turno
consumo_ideal = df_turno['consumo_medio_kwh'].min()
df_turno['oee_energetico'] = (consumo_ideal / df_turno['consumo_medio_kwh']).round(4)
df_turno['oee_pct']        = (df_turno['oee_energetico'] * 100).round(2)
df_turno['classificacao']  = pd.cut(
    df_turno['oee_pct'],
    bins=[0, 65, 75, 85, 100],
    labels=['Critico (<65%)', 'Abaixo (65-75%)', 'Aceitavel (75-85%)', 'Bom (>85%)']
)
print('OEE Energetico por Turno:')
print(df_turno[['turno','consumo_medio_kwh','oee_pct','classificacao']].to_string())

In [ ]:
# Metricas Diarias
df['data'] = pd.to_datetime(df['timestamp']).dt.date
df_diario = df.groupby('data').agg(
    consumo_kwh   = (energy_col, 'sum'),
    consumo_medio = (energy_col, 'mean'),
    registros     = (energy_col, 'count'),
    consumo_max   = (energy_col, 'max'),
).reset_index()
df_diario['data']             = pd.to_datetime(df_diario['data'])
df_diario                     = df_diario.sort_values('data')
df_diario['consumo_anterior'] = df_diario['consumo_kwh'].shift(1)
df_diario['variacao_pct']     = ((df_diario['consumo_kwh'] - df_diario['consumo_anterior']) / df_diario['consumo_anterior'] * 100).round(2)
df_diario['media_7d']         = df_diario['consumo_kwh'].rolling(7).mean().round(2)
print(f'Dias analisados: {len(df_diario)}')

In [ ]:
# Deteccao de Anomalias (Z-Score)
media  = df[energy_col].mean()
desvio = df[energy_col].std()
df['z_score']       = ((df[energy_col] - media) / desvio).round(4)
df['anomalia']      = (df['z_score'].abs() > 2.5).astype(int)
df['tipo_anomalia'] = 'Normal'
df.loc[df['z_score'] >  2.5, 'tipo_anomalia'] = 'Pico Alto'
df.loc[df['z_score'] < -2.5, 'tipo_anomalia'] = 'Queda Brusca'
anomalias = df[df['anomalia'] == 1]
print(f'Anomalias detectadas: {len(anomalias):,} ({len(anomalias)/len(df)*100:.2f}%)')
print(df.groupby('turno')['anomalia'].agg(['sum','mean']).round(4).to_string())

In [ ]:
# Metricas por Tipo de Carga
if load_col:
    load_c = load_col[0]
    df_carga = df.groupby(load_c).agg(
        registros     = (energy_col, 'count'),
        consumo_medio = (energy_col, 'mean'),
        consumo_total = (energy_col, 'sum'),
        anomalias     = ('anomalia', 'sum'),
    ).reset_index()
    df_carga['taxa_anomalia_pct'] = (df_carga['anomalias'] / df_carga['registros'] * 100).round(2)
    df_carga['pct_consumo']       = (df_carga['consumo_total'] / df_carga['consumo_total'].sum() * 100).round(2)
    print(df_carga.to_string())

In [ ]:
# Salva tabelas Gold
def salvar_gold(df_save, nome):
    path = os.path.join(GOLD_PATH, f'{nome}.parquet')
    pq.write_table(pa.Table.from_pandas(df_save), path, compression='snappy')
    print(f'Salvo: {nome}.parquet ({len(df_save):,} registros)')
    return path

paths = [
    salvar_gold(df,        'steel_gold_completo'),
    salvar_gold(df_turno,  'metricas_turno'),
    salvar_gold(df_diario, 'metricas_diario'),
    salvar_gold(anomalias, 'anomalias_detectadas'),
]
if load_col:
    paths.append(salvar_gold(df_carga, 'metricas_carga'))
print(f'{len(paths)} tabelas Gold salvas')

In [ ]:
# Upload Azure Gold
if AZURE_CONN:
    try:
        client = BlobServiceClient.from_connection_string(AZURE_CONN)
        container_client = client.get_container_client(CONTAINER)
        try:
            container_client.create_container()
        except Exception:
            pass
        for path in paths:
            blob_name = f'gold/{os.path.basename(path)}'
            with open(path, 'rb') as f:
                container_client.upload_blob(name=blob_name, data=f, overwrite=True)
            print(f'Upload: {blob_name}')
        print('Todas tabelas Gold enviadas para Azure!')
    except Exception as e:
        print(f'Erro Azure: {e}')
else:
    print('Azure nao configurado - tabelas salvas no Google Drive.')
print('Transformacao concluida!')